In [1]:
conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
pkgs <- c("Seurat", "CellChat", "SeuratObject", "scales", "farver")
for(p in pkgs) {
    if(file.exists(file.path(conda_lib, p))) {
        print(paste0(p, ": ", packageVersion(p, lib.loc = conda_lib)))
    } else {
        print(paste0(p, ": NOT FOUND"))
    }
}

[1] "Seurat: 5.2.1"
[1] "CellChat: 1.6.1"
[1] "SeuratObject: 5.2.0"
[1] "scales: 1.4.0"
[1] "farver: 2.1.2"


In [2]:
# 1. Lock the library path to ensure it uses your working environment
conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
.libPaths(conda_lib)

# 2. Quiet down R's verbose warnings and messages
options(warn = -1, message = FALSE)

# 3. Load libraries cleanly
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
})

# Reading in the barcodes & saving seurat objects (fractions + whole)

In [3]:
# 4. Define paths
data_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/seurat_transfer/"
output_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

# 5. Load the Matrix Market files exported from Scanpy
print("Reading MTX matrix and metadata...")
counts <- ReadMtx(
  mtx = file.path(data_dir, "matrix.mtx"),
  cells = file.path(data_dir, "barcodes.tsv"),
  features = file.path(data_dir, "features.tsv"),
  feature.column = 1,  # <--- Fixes the features.tsv error
  cell.column = 1      # <--- Ensures barcodes.tsv also reads correctly
)
# Load metadata and sync row names to match cell barcodes
metadata <- read.csv(file.path(data_dir, "metadata.csv"), row.names = 1)


[1] "Reading MTX matrix and metadata..."


In [4]:
# 6. Create the Universal Seurat Object
# This automatically places your raw MTX data into the 'counts' layer.
print("Creating universal Seurat object...")
seurat_global <- CreateSeuratObject(
  counts = counts, 
  meta.data = metadata, 
  project = "Xeno_Preclinical",
  assay = "RNA" # Standard default assay name
)

# Free up memory by removing the raw counts matrix (it is now safely inside the Seurat object)
rm(counts)
invisible(gc())

# 7. Perform Log-Normalization
# This calculates log1p(counts/10000) and places it into a NEW 'data' layer.
# Your raw counts remain perfectly intact in the 'counts' layer.
print("Normalizing data globally...")
seurat_global <- NormalizeData(
  seurat_global, 
  normalization.method = "LogNormalize", 
  scale.factor = 10000,
  verbose = FALSE  # Suppresses the progress bar to keep the notebook clean
)


[1] "Creating universal Seurat object..."
[1] "Normalizing data globally..."


In [5]:
# 8. Define your exact target output directory
output_dir <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

# 9. Save the WHOLE, universal Seurat object first
print("Saving the complete, universal Seurat object...")
whole_object_path <- file.path(output_dir, "seurat_whole_UMI_merged.rds")
saveRDS(seurat_global, file = whole_object_path)
print(paste0("Successfully saved whole object: ", whole_object_path))

# 10. Split the object into fractions by 'Condition'
print("Splitting object into fractions by Condition...")
seurat_fractions <- SplitObject(seurat_global, split.by = "Condition")

# Free up memory from the global object since it is now saved to disk
rm(seurat_global)
invisible(gc())

# 11. Cleanly save each individual condition fraction
target_conditions <- c(
  "Control",
  "Prematurely-terminated",
  "3month-protocol",
  "6month-protocol",
  "PCMV/PRV",
  "Rejection"
)

print("Saving individual fraction RDS files...")
for (cond in target_conditions) {
  if (cond %in% names(seurat_fractions)) {
    # Sanitize condition name for Linux file safety (e.g., PCMV/PRV -> PCMV_PRV)
    safe_name <- gsub("[/ ]", "_", cond)
    
    file_path <- file.path(output_dir, paste0("seurat_fraction_", safe_name, ".rds"))
    
    # Save the fractionated object
    saveRDS(seurat_fractions[[cond]], file = file_path)
    print(paste0("Successfully saved fraction: ", file_path))
  } else {
    print(paste0("Warning: Condition '", cond, "' was not found in the metadata."))
  }
}

print("All tasks completed successfully! Your environment is clean and ready for CellChat.")


[1] "Saving the complete, universal Seurat object..."
[1] "Successfully saved whole object: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_whole_UMI_merged.rds"
[1] "Splitting object into fractions by Condition..."
[1] "Saving individual fraction RDS files..."
[1] "Successfully saved fraction: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_Control.rds"
[1] "Successfully saved fraction: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_Prematurely-terminated.rds"
[1] "Successfully saved fraction: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_f